In [ ]:
!pip install RRAEsTorch

In [ ]:
import RRAEsTorch.config # Include this in all your scripts
from RRAEsTorch.AE_classes import *
from RRAEsTorch.training_classes import RRAE_Trainor_class, Trainor_class
from RRAEsTorch.utilities import get_data
import torch

In [ ]:
# Step 1: Get the data - replace this with your own data of the same shape.
all_errors = []
problem = "shift"
(
    x_train,
    x_test,
    p_train,
    p_test,
    y_train,
    y_test,
    pre_func_inp,
    pre_func_out,
    args,
) = get_data(problem)

print(f"Shape of data is {x_train.shape} (Ntr x T) and {x_test.shape} (Nte x T)")

In [ ]:
# Step 2: Specify the model to use, Strong_RRAE_MLP is ours (recommended).
method = "AE"

model_cls = Vanilla_AE_MLP

loss_type = "default"  # Specify the loss type, according to the model chosen.

In [ ]:
# Step 3: Specify the archietectures' parameters:
latent_size = 2  # latent space dimension

In [ ]:
# Step 4: Define your trainor, with the model, data, and parameters.
# Use RRAE_Trainor_class for the Strong RRAEs, and Trainor_class for other architetures.
trainor = Trainor_class(
    x_train,
    model_cls,
    latent_size=latent_size,
    in_size=x_train.shape[1],
    folder=f"{problem}/{method}_{problem}/",
    file=f"{method}_{problem}.pkl",
    norm_in="None",
    norm_out="None",
    kwargs_enc={
        "width_size": 300,
        "depth": 1,
    },
    kwargs_dec={
        "width_size": 300,
        "depth": 6,
    },
    out_train=x_train,
)

In [ ]:
# Step 5: Define the kw arguments for training. When using the Strong RRAE formulation,
# you need to specify training kw arguments (first stage of training with SVD to
# find the basis), and fine-tuning kw arguments (second stage of training with the
# basis found in the first stage).
device = "cuda" if torch.cuda.is_available() else "cpu"

training_kwargs = {
    "step_st": [2],  # Increase this to train well (e.g. 2000)
    "batch_size_st": [64, 64, 64, 64, 64],
    "lr_st": [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8],
    "print_every": 1,
    "loss_type": loss_type,
    "save_losses":True,
    "device": device,
}

ft_kwargs = {
    "step_st": [0], # Increase if you want to fine tune
    "batch_size_st": [64],
    "lr_st": [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8],
    "print_every": 1,
    "device": device,
}

In [ ]:
# Step 6: Train the model and get the predictions.
trainor.fit(
    x_train,
    y_train,
    input_val=x_train, # put your validation set here if you have one, otherwise None
    output_val=y_train, # put your validation set here if you have one, otherwise None
    #training_kwargs=training_kwargs,
    #ft_kwargs=ft_kwargs,
    pre_func_inp=pre_func_inp,
    pre_func_out=pre_func_out,
    **training_kwargs
)

# NOTE: the code does not overwrite the loss files, it gives every new file
# an index (e.g. all_losses_0, all_losses_1, etc.), if you run the model
# multiple times without deleting the loss file, consider changing idx
# below to specify the file of which training you want to plot
# trainor.plot_training_losses(idx=0) # to plot both training and validation losses


preds = trainor.evaluate(
   x_train, y_train, x_test, y_test, None, pre_func_inp, pre_func_out, device
)


trainor.save_model()

# Uncomment the following line if you want to hold the session to check your
# results in the console.

In [ ]:
# Some post processing tools:

alphas = trainor.model.get_coeffs(x_train)

# For vanilla AEs, this is similar to saying:
alphas0 = trainor.model.latent(x_train)
torch.allclose(alphas, alphas0)

In [ ]:
# now you can use the compressed alphas to visualize, etc. example:
import matplotlib.pyplot as plt
plt.scatter(alphas[:, 0].detach(), alphas[:, 1].detach())

In [ ]:
# to decode the compressed represetnation you can use:

pred = trainor.model.decode_coeffs(alphas)
pred.shape